# RAG + SQL: From Fundamentals to Advanced

This notebook teaches the two techniques behind your DRAM Express project — Retrieval-Augmented Generation (RAG) and Text-to-SQL — starting from first principles and building to the patterns you actually used (scoped views, few-shot prompting, sanitization, routing).

Every concept below is demonstrated with real, runnable code against your actual model and database, not just theory.

**Structure:**
1. SQL Fundamentals → Advanced
2. RAG Fundamentals → Advanced
3. Text-to-SQL: Why It's Hard, and How to Make It Reliable
4. Combining RAG + SQL: The Router Pattern

Run the setup cell first, then work through in order — later sections depend on the database and model being loaded.

## Setup

Loads the model (GPU-offloaded, same as your project) and connects to your existing `company.db`. Run this once.

In [2]:
import os
os.environ["LD_LIBRARY_PATH"] = "/home/bptremblay/miniconda3/envs/py311/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

import sqlite3
import time
from llama_cpp import Llama

MODEL_PATH = "/home/bptremblay/rag-sql-proj/models/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"
DB_PATH = "data/company.db"

llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=4096,
    n_threads=8,
    n_gpu_layers=-1,
    verbose=False,
)

def ask(prompt, system_prompt=None, temperature=0.0, max_tokens=300):
    """Simple helper used throughout this notebook."""
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})
    response = llm.create_chat_completion(messages=messages, temperature=temperature, max_tokens=max_tokens)
    return response["choices"][0]["message"]["content"]

print("Model loaded.")

Model loaded.


---
# Part 1: SQL Fundamentals → Advanced

Your project's database (`company.db`) is the playground for all of this.

## 1.1 The Basics: SELECT, WHERE, ORDER BY, LIMIT

Every SQL query answers a question about rows in a table. The core clauses, in the order SQL actually *executes* them (not the order you write them):

1. `FROM` — which table(s)
2. `WHERE` — filter rows
3. `SELECT` — which columns to return
4. `ORDER BY` — sort
5. `LIMIT` — cap how many rows come back

This execution order matters: it's why you can't reference a column alias from `SELECT` inside `WHERE` (WHERE runs before SELECT exists yet).

In [3]:
conn = sqlite3.connect(DB_PATH)

# Basic filter + sort + limit
rows = conn.execute("""
    SELECT first_name, last_name, signup_date
    FROM customers
    ORDER BY signup_date DESC
    LIMIT 5
""").fetchall()

for row in rows:
    print(row)

OperationalError: unable to open database file

## 1.2 Aggregation: COUNT, SUM, AVG, GROUP BY, HAVING

Aggregation collapses many rows into fewer summary rows. `GROUP BY` is "do this aggregation separately for each distinct value in this column." `HAVING` is like `WHERE`, but it filters *after* aggregation (since you can't use `WHERE` to filter on an aggregate result — `WHERE` runs before aggregation happens).

In [ ]:
# How many orders per status?
rows = conn.execute("""
    SELECT status, COUNT(*) as order_count
    FROM orders
    GROUP BY status
    ORDER BY order_count DESC
""").fetchall()

for row in rows:
    print(row)

In [ ]:
# HAVING example: only show statuses with more than 10 orders
rows = conn.execute("""
    SELECT status, COUNT(*) as order_count
    FROM orders
    GROUP BY status
    HAVING COUNT(*) > 10
""").fetchall()

for row in rows:
    print(row)

# Try changing WHERE COUNT(*) > 10 above to HAVING -- it will error.
# WHERE can't see aggregate results because it runs before GROUP BY.

## 1.3 JOINs

A JOIN combines rows from two tables based on a matching condition — almost always a foreign key relationship (like `orders.customer_id` matching `customers.customer_id`).

**INNER JOIN** — only rows that match in both tables.
**LEFT JOIN** — all rows from the left table, plus matches from the right (NULL where there's no match).

The difference matters a lot: if you want "every customer, including ones with zero orders," INNER JOIN would silently drop them. This is one of the most common real-world SQL bugs.

In [ ]:
# INNER JOIN: customers WITH orders (customers with no orders are excluded)
rows = conn.execute("""
    SELECT c.first_name, o.order_id, o.status
    FROM customers c
    INNER JOIN orders o ON c.customer_id = o.customer_id
    LIMIT 5
""").fetchall()
print("INNER JOIN:")
for row in rows:
    print(" ", row)

In [ ]:
# LEFT JOIN: ALL customers, even ones with zero orders (order_id will be None for them)
rows = conn.execute("""
    SELECT c.first_name, o.order_id
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    WHERE o.order_id IS NULL
""").fetchall()
print(f"Customers with ZERO orders (found via LEFT JOIN): {len(rows)}")
for row in rows[:5]:
    print(" ", row)

# This is exactly the "why does Christopher have no orders" mystery from earlier in your project --
# a LEFT JOIN like this is how you'd have found that answer directly in SQL.

## 1.4 Subqueries vs JOINs vs CTEs

Three ways to combine logic, roughly in order of readability:

- **Subquery** — a query nested inside another. Fine for simple cases, gets hard to read when nested deeply.
- **JOIN** — usually more efficient and more readable than an equivalent subquery for combining tables.
- **CTE (Common Table Expression, `WITH ... AS`)** — names a subquery so you can reference it like a temporary table. Best for readability when logic gets complex, and can reference itself for recursive queries.

All three can often express the *same* logic — the choice is about clarity and performance, not capability.

In [ ]:
# Same question, three ways: "customers whose most recent order is a payment_dispute"

# 1) Subquery
rows = conn.execute("""
    SELECT first_name FROM customers
    WHERE customer_id IN (
        SELECT customer_id FROM orders
        WHERE status = 'payment_dispute'
    )
    LIMIT 5
""").fetchall()
print("Subquery:", rows)

# 2) JOIN
rows = conn.execute("""
    SELECT DISTINCT c.first_name FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    WHERE o.status = 'payment_dispute'
    LIMIT 5
""").fetchall()
print("JOIN:", rows)

# 3) CTE
rows = conn.execute("""
    WITH disputed AS (
        SELECT DISTINCT customer_id FROM orders WHERE status = 'payment_dispute'
    )
    SELECT c.first_name FROM customers c
    JOIN disputed d ON c.customer_id = d.customer_id
    LIMIT 5
""").fetchall()
print("CTE:", rows)

## 1.5 Views (You Already Built These)

A `VIEW` is a saved query that behaves like a table. Every time you query it, it re-runs the underlying query live — it doesn't store a copy of the data.

This is exactly what `auth.py`'s `my_orders`, `my_order_items`, and `my_reviews` are: views pre-filtered to one customer, so any query against them is automatically scoped — no matter what the LLM generates on top, it structurally cannot see another customer's rows.

In [ ]:
# Recreate the pattern from your project, for a specific customer, as a demo
conn.execute("DROP VIEW IF EXISTS demo_my_orders")
conn.execute("""
    CREATE TEMP VIEW demo_my_orders AS
    SELECT order_id, product_id, order_date, status, installments_remaining
    FROM orders WHERE customer_id = 4
""")

# Now query the VIEW like a normal table -- no customer_id needed or even visible
rows = conn.execute("SELECT * FROM demo_my_orders").fetchall()
for row in rows:
    print(row)

# This is the core security idea from your project: the filtering lives in the
# view definition, not in something the LLM has to remember to add.

## 1.6 Indexes and Query Performance

Without an index, SQLite has to scan every row in a table to find matches (a "full table scan"). An index is a separate, sorted data structure that lets it jump straight to matching rows instead — the same idea as a book's index letting you skip to a page instead of reading cover to cover.

`EXPLAIN QUERY PLAN` shows you what SQLite actually plans to do — whether it'll use an index or scan the whole table.

In [ ]:
# Check the query plan for a filter on a column with NO index
plan = conn.execute("""
    EXPLAIN QUERY PLAN
    SELECT * FROM orders WHERE status = 'payment_dispute'
""").fetchall()
print("Before index:")
for row in plan:
    print(" ", row)
# You'll likely see "SCAN orders" -- it's checking every row

In [ ]:
# Now add an index on that column
conn.execute("CREATE INDEX IF NOT EXISTS idx_orders_status ON orders(status)")

plan = conn.execute("""
    EXPLAIN QUERY PLAN
    SELECT * FROM orders WHERE status = 'payment_dispute'
""").fetchall()
print("After index:")
for row in plan:
    print(" ", row)
# Now you should see "SEARCH orders USING INDEX idx_orders_status" -- it jumps
# straight to matching rows instead of scanning everything.

# On a table with 100 rows this makes almost no measurable difference.
# On a table with 10 million rows, it's the difference between milliseconds and minutes.

## 1.7 Window Functions

Window functions compute something *across a set of related rows* without collapsing them into one row the way `GROUP BY` does. Classic use case: ranking.

`ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)` numbers rows within groups, e.g. "for each customer, number their orders from newest to oldest".

In [ ]:
# For each customer, rank their orders from newest to oldest
rows = conn.execute("""
    SELECT customer_id, order_id, order_date,
           ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date DESC) as order_rank
    FROM orders
    ORDER BY customer_id, order_rank
    LIMIT 10
""").fetchall()
for row in rows:
    print(row)

# order_rank = 1 means "this customer's most recent order" -- useful for
# "what's my LATEST order" style questions, without a separate subquery per customer.

## 1.8 Why Your Schema Looks the Way It Does (Normalization)

Your `seed_db.py` schema splits data into `customers`, `products`, `orders`, `order_items` rather than one giant flat table. This is **normalization** — avoiding storing the same fact in multiple places.

If you stored the customer's name directly on every order row instead of just a `customer_id`, and that customer's name ever needed correcting, you'd have to update it in every single order row. With normalization, it's stored once in `customers` and referenced everywhere else — one source of truth.

The tradeoff: normalized data needs JOINs to reassemble the full picture, which costs some performance. For a support chatbot's scale, that tradeoff is trivially worth it.

---
# Part 2: RAG Fundamentals → Advanced

Your `build_index.py` and `test_retrieval.py` already implement everything below — here's *why* each piece works the way it does.

## 2.1 Why RAG Exists

An LLM's knowledge is frozen at training time and general-purpose. It has never seen your `return-policy.md` — DRAM Express doesn't exist in its training data. Two ways to fix that:

- **Fine-tuning** — retrain the model on your data. Expensive, slow, and the model "forgets" old knowledge as it learns new. Overkill for most use cases.
- **RAG** — leave the model alone, and just hand it the relevant text *at the moment it answers*, as part of the prompt. Cheap, instant to update (just re-index), and the model's original abilities stay intact.

RAG is "open-book exam" instead of "memorize everything."

## 2.2 Embeddings: How "Meaning" Becomes Numbers

An embedding model converts text into a list of numbers (a vector) such that texts with similar *meaning* end up as similar *numbers* — even if they don't share any of the same words.

Let's prove that with your actual embedding model.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "Can I get my money back?",
    "What is your refund policy?",
    "Do you ship to Florida?",
    "The weather is nice today.",
]

vectors = embed_model.encode(sentences)
print(f"Each sentence became a vector of {vectors.shape[1]} numbers.")
print(f"First 8 numbers of sentence 0's vector: {vectors[0][:8]}")

In [ ]:
# Cosine similarity: measures the ANGLE between two vectors (1.0 = identical
# meaning, 0 = unrelated, -1 = opposite). This is what powers retrieval --
# ChromaDB is doing this same math under the hood for every stored chunk.

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("'refund' vs 'money back' (different words, same meaning):")
print(" ", cosine_similarity(vectors[0], vectors[1]))

print("'refund' vs 'ship to Florida' (different topic):")
print(" ", cosine_similarity(vectors[0], vectors[2]))

print("'refund' vs 'weather' (totally unrelated):")
print(" ", cosine_similarity(vectors[0], vectors[3]))

# Notice sentence 0 and 1 share almost no words in common, yet score highest --
# the model captured MEANING, not just keyword overlap. This is exactly why
# RAG retrieval can find "how do I get a refund" when the doc says "returns and refunds".

## 2.3 Chunking Strategies

You can't embed an entire document as one vector and expect precise retrieval — a whole document's "average meaning" is too vague to match a specific question well. So documents get split into **chunks** first.

Trade-offs you already hit in your project:
- **Too large:** a chunk's embedding becomes a blurry average of everything in it, and irrelevant content drowns out the relevant part (your original 500-char chunks merging "What's Covered" with "Filing a Claim").
- **Too small:** you lose surrounding context, and a chunk might not contain enough information to actually answer anything on its own (your "Can I get a refund?" test returning just the bare doc header).
- **Overlap:** carrying a bit of the previous chunk forward prevents an important sentence from being orphaned right at a boundary — but naive character-based overlap can cut mid-word (the bug you fixed with `_word_boundary_tail`).

There's no universally "correct" chunk size — it depends on how information-dense your documents are.

In [ ]:
# Demonstrate the chunk-size tradeoff directly
sample_doc = '''
# Warranty Policy

## The Guarantee
If you are dissatisfied within 24 hours, fill out our form.
Ratings below 5 stars qualify for a refund.

## What's Covered
We cover manufacturing defects and delivery errors.
We do not cover buyer's remorse.

## Filing a Claim
Claims must be submitted within the 24 hour window.
'''

def naive_chunk(text, size):
    return [text[i:i+size] for i in range(0, len(text), size)]

print("Chunk size 80 (too small -- sections get sliced mid-thought):")
for c in naive_chunk(sample_doc.strip(), 80)[:2]:
    print(repr(c[:80]))

print("\nChunk size 250 (closer to one section per chunk):")
for c in naive_chunk(sample_doc.strip(), 250)[:2]:
    print(repr(c[:100]) + "...")

## 2.4 What ChromaDB Is Actually Doing

When you call `collection.query()`, ChromaDB isn't checking every stored vector one by one against your query (that would be slow at scale — this is called "brute force" or "exact" search). Production vector databases use **Approximate Nearest Neighbor (ANN)** algorithms — ChromaDB uses one called **HNSW** (Hierarchical Navigable Small World) by default.

The core idea: build a graph structure over your vectors at insert time, so at query time you can navigate through it and land near the best matches in a handful of hops, instead of comparing against every single vector. You trade a small amount of accuracy (it's *approximate*, hence the name) for massive speed gains at scale — for your ~30 chunks this doesn't matter at all, but it's exactly why services like Pinecone or Chroma can search millions of vectors in milliseconds.

## 2.5 Evaluating Retrieval Quality: Precision and Recall @ k

When you tested `test_retrieval.py` by eyeballing whether the right doc came back, you were doing an informal version of a real evaluation methodology:

- **Precision@k** — of the top *k* results returned, what fraction were actually relevant?
- **Recall@k** — of all the relevant chunks that exist, what fraction did you actually retrieve in the top *k*?

These can trade off against each other. Increasing `n_results` (k) tends to increase recall (you're less likely to miss the right chunk) but can decrease precision (more irrelevant chunks come along for the ride) — exactly the tension you navigated when choosing `n_results=1` vs `2` earlier.

In [ ]:
# A tiny manual precision@k check, using your actual retrieval index
import chromadb

client = chromadb.PersistentClient(path="data/chroma_db")
collection = client.get_collection("dram_express_docs")

test_cases = [
    ("Can I get a refund?", "warranty.md"),   # (question, expected correct source file)
    ("Do you ship to Florida?", "shipping.md"),
    ("What happens if I miss a payment?", "installment-terms.md"),
]

for question, expected_source in test_cases:
    q_embedding = embed_model.encode([question]).tolist()
    results = collection.query(query_embeddings=q_embedding, n_results=1)
    actual_source = results["metadatas"][0][0]["source"]
    correct = "CORRECT" if actual_source == expected_source else "WRONG"
    print(f"[{correct}] Q: {question!r} -> got {actual_source}, expected {expected_source}")

## 2.6 Hybrid Search (Beyond Pure Embeddings)

Pure embedding search is great at *meaning* but sometimes weak on exact terms — e.g. an order number, product SKU, or exact phrase. Production RAG systems often combine embedding search with traditional keyword search (commonly an algorithm called **BM25**, a refined version of "count matching words, weighted by how rare/important they are") and merge the two result sets.

Your project doesn't do this (it's pure embedding search), which is fine at this scale — but it's worth knowing this is the next lever production systems pull when pure semantic search misses exact-match queries.

## 2.7 RAG Failure Modes You Already Hit

You personally encountered these firsthand:

1. **Chunk boundary misses** — the right information exists in the docs, but got split across a boundary and no single chunk fully answers the question (your `warranty.md` "Filing a Claim" vs "What's Covered" mix-up).
2. **Lexical false positives** — a chunk scores as a top match because it shares a *word* with the question, not the *meaning* (your "Is my payment information secure?" chunk matching "missed a payment" questions).
3. **Retrieval succeeds, generation still fails** — the right chunk comes back, but the model still hallucinates or hedges oddly if the chunk is too thin (your "refund policy" answer that hedged after retrieving just a bare header).
4. **No relevant content exists at all** — if nobody wrote a doc about something, no amount of retrieval tuning will produce a correct answer (why you had to write `products.md` before "what packages do you offer" could work).

Case 4 is the most important lesson: **RAG can only retrieve what was written down.** It's not a substitute for actually documenting the thing you want answered.

---
# Part 3: Text-to-SQL — Why It's Hard, and How to Make It Reliable

This is the part of your project that broke the most in interesting ways, and for good reason: it's genuinely one of the harder LLM application problems.

## 3.1 Why Text-to-SQL Is Hard

Unlike RAG (where a wrong retrieval usually just gives a vague answer), a wrong SQL query can:
- Return confidently *wrong* data that looks correct (a subtle JOIN mistake, an off-by-one in a filter)
- Fail outright with a syntax/column error
- In the worst case, if unconstrained, modify or delete data it shouldn't touch

The model has to correctly do **schema linking** — matching natural language ("my payments") to the right table/column (`installments_remaining`) — with zero ambiguity tolerance, since SQL either runs or it doesn't; there's no partial credit.

## 3.2 Prompt Engineering for SQL Generation

Three techniques you used, and why each one matters:

**1. Give it the schema, precisely.** The model can't query a table it doesn't know exists. Comments on ambiguous columns (like listing valid `status` values) reduce guessing.

**2. Constrain output format strictly.** Telling it "output ONLY the SQL, no explanation" is necessary because a chat-tuned model's default instinct is to be helpful and explain itself — which breaks programmatic parsing of the response.

**3. Few-shot examples.** This was the single most effective fix in your project — showing 3-4 worked examples of question→SQL pairs stopped the model from inventing unnecessary subqueries far more reliably than just describing the schema in prose. Let's prove that side by side.

In [ ]:
SCHEMA = '''
Tables:
my_orders (order_id, product_id, order_date, status, installments_remaining)
my_reviews (review_id, order_id, rating, review_text)
'''

# WITHOUT few-shot examples
prompt_no_examples = f'''You are a SQL generator. Output ONLY the SQL query.
Schema:
{SCHEMA}
'''

# WITH few-shot examples
prompt_with_examples = prompt_no_examples + '''
Examples:
Q: What are my reviews?
A: SELECT rating, review_text FROM my_reviews;

Q: How many payments do I have left?
A: SELECT SUM(installments_remaining) FROM my_orders;
'''

question = "What did I say in my review?"

print("WITHOUT few-shot examples:")
print(ask(question, system_prompt=prompt_no_examples, max_tokens=100))

print("\nWITH few-shot examples:")
print(ask(question, system_prompt=prompt_with_examples, max_tokens=100))

# Run this a few times -- the few-shot version should consistently produce a
# simpler, more direct query. This is the exact fix that resolved your
# "read my review" bug in router.py.

## 3.3 Security: Why Prompt Instructions Aren't Enough

You discovered this the hard way: telling the model "don't filter by customer_id, you don't know the real value" reduced the problem, but the model kept guessing `customer_id = 1` anyway — and since that guess didn't match the real logged-in customer, it silently zeroed out correct results instead of erroring.

**The structural fix you landed on:** the `customer_id` column was removed from the views entirely. Not hidden, not discouraged — *absent*. If the model tries to reference it, SQLite throws a clear "no such column" error instead of silently returning wrong data.

This is a specific instance of a general security principle: **never rely on an LLM's compliance for something a permission boundary should enforce.** The model is a text predictor, not a rule-follower with guarantees — treat its outputs the way you'd treat any untrusted user input.

In [ ]:
# Demonstrate: even with a firm instruction, watch it guess anyway (may vary by run)
tempting_schema = '''
Tables:
my_orders (order_id, customer_id, product_id, status)
-- customer_id column EXISTS here, unlike your real project's fixed version
'''

prompt = f'''You are a SQL generator. Output ONLY the SQL.
IMPORTANT: Do NOT filter by customer_id, it is already handled, you do not know its value.
Schema:
{tempting_schema}
'''

result = ask("What's the status of my orders?", system_prompt=prompt, max_tokens=100)
print(result)
print()
print("If 'customer_id' shows up in that query despite the explicit instruction,")
print("that's the exact failure mode your project hit -- proving the fix has to be")
print("structural (remove the column), not just instructional.")

## 3.4 Sanitization as a Safety Net

Even with a good schema and good instructions, treat the model's SQL as untrusted output and defend at the code layer too — belt and suspenders, same as your `sanitize_sql()` regex that strips any `customer_id = N` clause before execution, regardless of whether the prompt-level fix worked.

General principle for any LLM-generated code that will actually execute: validate/sanitize before running it, don't just trust that good prompting worked this time.

## 3.5 Self-Correction Loops (A Pattern Your Project Doesn't Use, But Could)

A more advanced pattern: if generated SQL throws an error, feed the error message *back* to the model and ask it to fix its own query, rather than just failing. Let's build a minimal version of this.

In [ ]:
def nl_to_sql_with_retry(question, schema, max_retries=2):
    system_prompt = f"""You are a SQL generator for SQLite. Output ONLY the SQL query.
Schema:
{schema}
"""
    sql = ask(question, system_prompt=system_prompt, max_tokens=150)

    for attempt in range(max_retries):
        try:
            conn.execute(f"EXPLAIN {sql}")  # validates syntax without running it
            return sql, attempt
        except sqlite3.Error as e:
            correction_prompt = f"""Your previous SQL query had an error.
Query: {sql}
Error: {e}
Output ONLY a corrected SQL query.
Schema:
{schema}
Question: {question}
"""
            sql = ask(correction_prompt, max_tokens=150)

    return sql, max_retries

schema = "orders (order_id, customer_id, product_id, order_date, status, installments_remaining)"
sql, attempts = nl_to_sql_with_retry("How many orders are shipped", schema)
print(f"Final SQL after {attempts} correction(s): {sql}")

# This pattern -- generate, validate, self-correct on error -- is how more
# advanced text-to-SQL systems (and coding agents generally) get meaningfully
# higher success rates than a single generate-and-hope attempt.

---
# Part 4: Combining RAG + SQL — The Router Pattern

Your `router.py` ties everything above together. Here's the pattern in isolation.

## 4.1 Why You Need a Router At All

RAG answers "what's true in general" (static documents). Text-to-SQL answers "what's true for this specific record right now" (structured, queryable data). Neither can answer the other's questions well:

- Asking RAG "what's the status of order #22" — there's no document containing that, it's data, not prose.
- Asking SQL "what's your return policy" — there's no table containing your written policy as structured rows.

The router's job is a **classification problem**: given a question, which source(s) can actually answer it?

In [ ]:
ROUTER_PROMPT = '''You classify questions as needing "sql" (personal account data),
"rag" (general policy info), or "both". Respond with exactly one word.'''

test_questions = [
    "What's your return policy?",
    "What's the status of order 22?",
    "Can I get a refund on my Basic tier order?",  # needs both
]

for q in test_questions:
    label = ask(q, system_prompt=ROUTER_PROMPT, max_tokens=5).strip().lower()
    print(f"{label:6s} <- {q}")

## 4.2 Why "Both" Is the Interesting Case

When a question needs both sources, the router gathers context from each independently, then hands *both* pieces of context to a final synthesis call. The model's job at that final step isn't retrieval or SQL generation anymore — it's pure composition: weaving two facts into one coherent answer.

This is the same "combined two separate chunks into one sentence" behavior you noticed earlier when testing your project — proof the model was actually synthesizing, not just echoing retrieved text verbatim.

## 4.3 Guardrails Belong at the Synthesis Layer, Scoped Precisely

Your project's biggest guardrail lesson: a broadly-worded refusal instruction ("don't say anything harmful") ended up blocking the assistant from relaying a customer's *own* review back to them, just because it contained profanity. The fix wasn't removing the guardrail — it was scoping it precisely: the rule applies to what the *customer is asking the model to do*, not to *retrieved data being reported factually*.

General lesson for any guardrail you write: state exactly what triggers it and what doesn't, or you'll get both false negatives (jailbreaks slipping through) and false positives (legitimate requests refused) at unpredictable rates.

---
## Summary: The Full Stack

| Layer | Concept | Your project's implementation |
|---|---|---|
| Storage | Normalized relational schema | `customers`, `products`, `orders`, `order_items`, `reviews` |
| Security | Structural scoping, not prompted trust | Customer-scoped SQL views with `customer_id` removed entirely |
| Retrieval | Chunking + embeddings + ANN search | `build_index.py`, ChromaDB, `all-MiniLM-L6-v2` |
| Generation | Constrained, few-shot prompted LLM calls | `router.py`'s `SQL_GEN_SYSTEM_PROMPT`, `ANSWER_SYSTEM_PROMPT` |
| Safety net | Sanitize untrusted model output before executing | `sanitize_sql()` |
| Orchestration | Classification-based routing | `classify_question()` + merged context synthesis |

You built a genuinely complete version of this stack, and — more importantly — you hit and fixed the real failure modes that make this hard in practice, not just the happy path.